In [1]:
import os, gc, glob, hashlib
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DATASET_ROOT = os.environ.get("DATASET_ROOT", "/home/jovyan/shared/donut/Alzheimer/OAS1")
SEEDS = [1334, 0, 1, 42, 2024, 3407]
SPLIT_PATH = "split.json"   # ไฟล์กลางที่ล็อกไว้
CHECK_ID = "0454"

SIZE_2D = 224
SHAPE_3D = (128, 128, 96)
MODES = ["2D", "2.5D", "3D"]
EXPECTED = {"2D": (1, SIZE_2D, SIZE_2D), "2.5D": (3, SIZE_2D, SIZE_2D), "3D": (1,) + SHAPE_3D}


In [2]:
df = pd.read_excel(f"{DATASET_ROOT}/_oasis_cross-sectional.xlsx")
df = df[df["CDR"].notna()]
df["label"] = (df["CDR"] > 0).astype(int)

def find_file(subject_id):
    pattern = f"{DATASET_ROOT}/{subject_id}/PROCESSED/MPRAGE/T88_111/*_masked_gfc.hdr"
    files = glob.glob(pattern)
    return files[0] if files else None

df["path"] = df["ID"].apply(find_file)
df = df[df["path"].notna()]
df["subject_id"] = df["ID"].apply(lambda x: x.split("_")[1])
print("df หลังกรอง:", df.shape, "| subject ไม่ซ้ำ:", df["subject_id"].nunique(), "|", CHECK_ID, "อยู่ใน df:", CHECK_ID in set(df["subject_id"]))


df หลังกรอง: (234, 15) | subject ไม่ซ้ำ: 234 | 0454 อยู่ใน df: True


In [3]:
import random
try:
    import torch
except ImportError:
    torch = None

def load_and_prep(path):
    # crop สมอง + normalize เหมือนโน้ตบุ๊กจริง (ยังไม่ resize)
    v = nib.load(path).get_fdata()
    if v.ndim == 4:
        v = v[:, :, :, 0]
    nz = np.argwhere(v > 0)
    lo, hi = nz.min(axis=0), nz.max(axis=0)
    v = v[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1]
    p_lo, p_hi = np.percentile(v, [1, 99])
    v = np.clip(v, p_lo, p_hi)
    return (v - p_lo) / (p_hi - p_lo + 1e-8)

def resize2d(s):
    return zoom(s, (SIZE_2D / s.shape[0], SIZE_2D / s.shape[1]), order=1)

def to_target(v, mode):
    if mode == "3D":
        return zoom(v, [t / c for t, c in zip(SHAPE_3D, v.shape)], order=1)[None]       # (1,128,128,96)
    if mode == "2D":
        return resize2d(v[:, :, v.shape[2] // 2])[None]                                  # (1,224,224)
    if mode == "2.5D":
        cx, cy, cz = [s // 2 for s in v.shape]
        return np.stack([resize2d(v[cx, :, :]), resize2d(v[:, cy, :]), resize2d(v[:, :, cz])])  # (3,224,224)
    raise ValueError(mode)

def checksum(a):
    return hashlib.sha1(np.ascontiguousarray(a).tobytes()).hexdigest()[:10]

def build_cache(mode):
    # อ่านและประมวลผลแต่ละไฟล์ครั้งเดียว (ผลไม่ขึ้นกับ seed)
    cache = {}
    for sid, path in tqdm(zip(df["subject_id"], df["path"]), total=len(df), desc=f"โหลด {mode}"):
        cache[sid] = to_target(load_and_prep(path), mode).astype(np.float32)
    return cache

def assemble(cache, ids):
    ids = [s for s in df["subject_id"] if s in ids]      # ลำดับเดียวกับ df
    return np.stack([cache[s] for s in ids]), ids


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


In [4]:
import json

def load_split(path=SPLIT_PATH):
    """อ่าน split.json -> (train_ids, val_ids, test_ids) พร้อมตรวจความถูกต้อง"""
    with open(path) as f:
        sp = json.load(f)
    a, b, c = set(sp["train"]), set(sp["val"]), set(sp["test"])
    assert not (a & b) and not (a & c) and not (b & c), "มีคนซ้ำระหว่างชุด"
    ids_df = set(df["subject_id"])
    assert (a | b | c) == ids_df, "รายชื่อใน df ไม่ตรงกับ split.json"
    lab = df.groupby("subject_id")["label"].first()
    assert all(int(lab[s]) == int(sp["labels"][s]) for s in ids_df), "label ไม่ตรง"
    return a, b, c

REF = load_split()          # ค่าอ้างอิงจากไฟล์ (อ่านครั้งเดียว ก่อนวน seed)
print("อ่าน", SPLIT_PATH, "-> train/val/test =", [len(x) for x in REF])

def assemble(cache, ids):
    ids = [s for s in df["subject_id"] if s in ids]      # ลำดับเดียวกับ df
    return np.stack([cache[s] for s in ids]), ids


อ่าน split.json -> train/val/test = [163, 35, 36]


In [5]:
records = []
img_ck = {}          # (mode, subject) -> {seed: checksum}

for mode in MODES:
    print("#" * 70)
    print(f"โมเดล {mode}  target shape = {EXPECTED[mode]}")
    cache = build_cache(mode)

    for sd in SEEDS:
        set_seed(sd)                          # seed เปลี่ยน แต่แหล่งรายชื่อคือไฟล์เดิม
        a, b, c = load_split()                # โหลดใหม่จากไฟล์ทุกรอบ
        x_tr, s_tr = assemble(cache, a)
        x_va, s_va = assemble(cache, b)
        x_te, s_te = assemble(cache, c)
        for nm, x in [("train", x_tr), ("val", x_va), ("test", x_te)]:
            assert x.shape[1:] == EXPECTED[mode], f"{mode} {nm} shape ผิด: {x.shape}"

        same_as_file = (a == REF[0]) and (b == REF[1]) and (c == REF[2])
        for nm, sids, x in [("train", s_tr, x_tr), ("val", s_va, x_va), ("test", s_te, x_te)]:
            for i, s in enumerate(sids):
                img_ck.setdefault((mode, s), {})[sd] = checksum(x[i])
        where = next((nm for nm, sids in [("train", s_tr), ("val", s_va), ("test", s_te)] if CHECK_ID in sids), "ไม่พบ")
        split_hash = hashlib.sha1("|".join(",".join(sorted(s)) for s in (a, b, c)).encode()).hexdigest()[:10]
        print(f"  {mode:5s} seed={sd:<5} train={len(s_tr)} val={len(s_va)} test={len(s_te)} | ตรงกับไฟล์={same_as_file} | {CHECK_ID} -> {where}")
        records.append({"mode": mode, "seed": sd, "same_as_file": same_as_file,
                        "where": where, "split_hash": split_hash})
        del x_tr, x_va, x_te
    del cache
    gc.collect()


######################################################################
โมเดล 2D  target shape = (1, 224, 224)


โหลด 2D: 100%|██████████| 234/234 [00:12<00:00, 18.07it/s]


  2D    seed=1334  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2D    seed=0     train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2D    seed=1     train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2D    seed=42    train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2D    seed=2024  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2D    seed=3407  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
######################################################################
โมเดล 2.5D  target shape = (3, 224, 224)


โหลด 2.5D: 100%|██████████| 234/234 [00:12<00:00, 18.45it/s]


  2.5D  seed=1334  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2.5D  seed=0     train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2.5D  seed=1     train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2.5D  seed=42    train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2.5D  seed=2024  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  2.5D  seed=3407  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
######################################################################
โมเดล 3D  target shape = (1, 128, 128, 96)


โหลด 3D: 100%|██████████| 234/234 [00:19<00:00, 12.03it/s]


  3D    seed=1334  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  3D    seed=0     train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  3D    seed=1     train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  3D    seed=42    train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  3D    seed=2024  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val
  3D    seed=3407  train=163 val=35 test=36 | ตรงกับไฟล์=True | 0454 -> val


In [6]:
summary = pd.DataFrame(records)
print(summary.to_string(index=False))
print()

ok1 = bool(summary["same_as_file"].all())
print("1) ทุก (โมเดล, seed) ได้รายชื่อ train/val/test ตรงกับ split.json เป๊ะ:", ok1)

ok2 = summary["split_hash"].nunique() == 1
print("2) split_hash มีค่าเดียวทั้ง", len(summary), "การรัน (ข้าม seed และข้ามโมเดล):", ok2)

ok3 = summary.groupby("mode")["where"].nunique().eq(1).all() and summary["where"].nunique() == 1
print(f"3) {CHECK_ID} อยู่ชุดเดิมทุก seed ทุกโมเดล:", ok3, "->", summary["where"].unique().tolist())

bad = [k for k, d in img_ck.items() if len(set(d.values())) != 1]
print(f"4) ภาพของทุกคนเหมือนเดิมทุก seed (checksum): {not bad}  (ตรวจ {len(img_ck)} คู่ โมเดล-คน)")

print()
print("ผ่านทั้งหมด" if (ok1 and ok2 and ok3 and not bad) else "มีบางข้อไม่ผ่าน ดูตารางด้านบน")


mode  seed  same_as_file where split_hash
  2D  1334          True   val 67c0d18d36
  2D     0          True   val 67c0d18d36
  2D     1          True   val 67c0d18d36
  2D    42          True   val 67c0d18d36
  2D  2024          True   val 67c0d18d36
  2D  3407          True   val 67c0d18d36
2.5D  1334          True   val 67c0d18d36
2.5D     0          True   val 67c0d18d36
2.5D     1          True   val 67c0d18d36
2.5D    42          True   val 67c0d18d36
2.5D  2024          True   val 67c0d18d36
2.5D  3407          True   val 67c0d18d36
  3D  1334          True   val 67c0d18d36
  3D     0          True   val 67c0d18d36
  3D     1          True   val 67c0d18d36
  3D    42          True   val 67c0d18d36
  3D  2024          True   val 67c0d18d36
  3D  3407          True   val 67c0d18d36

1) ทุก (โมเดล, seed) ได้รายชื่อ train/val/test ตรงกับ split.json เป๊ะ: True
2) split_hash มีค่าเดียวทั้ง 18 การรัน (ข้าม seed และข้ามโมเดล): True
3) 0454 อยู่ชุดเดิมทุก seed ทุกโมเดล: True -> ['val']
4